In [ ]:
!pip install ultralytics opencv-python pyyaml

In [2]:
import cv2
import numpy as np
import yaml

# Load class names from a YAML file (YOLO format)
def load_classes(path):
    try:
        with open(path, 'r') as f:
            # The YAML should contain a key 'names' that maps to a list of class names
            return yaml.safe_load(f).get('names', [])
    except:
        # Default fallback classes if YAML fails to load
        return ['person', 'car', 'chair', 'bottle', 'bird', 'sofa', 'cycle', 'horse', 'bus']

# Detect objects in the frame using the YOLO model
def detect_objects(net, frame, size=640, conf_th=0.4, nms_th=0.45):
    h, w = frame.shape[:2]  # Original frame dimensions (height, width)

    # Preprocess the image: scale pixel values, resize, convert RGB -> BGR, etc.
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (size, size), swapRB=True, crop=False)

    net.setInput(blob)          # Pass the blob to the neural network
    outputs = net.forward()[0]  # Get the first output from the network (YOLO returns a 3D array)

    boxes, confidences, class_ids = [], [], []  # Lists to store results

    # Iterate over all predictions
    for det in outputs:
        # YOLO format: [center_x, center_y, width, height, object_confidence, class1_score, class2_score, ...]
        score = det[4] * np.max(det[5:])  # Final score = objectness × max class confidence
        if score > conf_th:
            cx, cy, bw, bh = det[:4]  # center x, center y, width, height
            x = int((cx - bw/2) * w / size)  # Convert to top-left x in original frame
            y = int((cy - bh/2) * h / size)  # Convert to top-left y
            boxes.append([x, y, int(bw * w / size), int(bh * h / size)])  # Convert width/height to original scale
            confidences.append(float(score))  # Save confidence
            class_ids.append(np.argmax(det[5:]))  # Save class index with highest score

    # Apply Non-Maximum Suppression to remove overlapping boxes
    idxs = cv2.dnn.NMSBoxes(boxes, confidences, conf_th, nms_th)
    return boxes, confidences, class_ids, idxs

# Draw bounding boxes and class labels on the frame
def draw_detections(frame, boxes, confidences, class_ids, indices, names):
    for i in indices.flatten() if len(indices) else []:
        x, y, w, h = boxes[i]
        label = f"{names[class_ids[i]]} {confidences[i]:.2f}" if class_ids[i] < len(names) else f"class_{class_ids[i]}"
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)  # Draw bounding box in green
        cv2.putText(frame, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)  # Draw label above box

# Main function to load the model and run webcam detection
def main():
    # Paths to model and YAML config file
    model = r'C:\data_science\Image\Model2\weights\best.onnx'
    yaml_file = r'C:\data_science\Image\data.yaml'

    # Load class names from YAML file
    class_names = load_classes(yaml_file)

    # Load the ONNX model
    net = cv2.dnn.readNetFromONNX(model)

    # Set backend and target device (CPU here, could be CUDA if available)
    net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
    net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

    # Open webcam (device 0 is usually the default camera)
    cap = cv2.VideoCapture(0)

    # Main loop to read and process frames
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Run detection
        boxes, confidences, class_ids, indices = detect_objects(net, frame)

        # Draw results
        draw_detections(frame, boxes, confidences, class_ids, indices, class_names)

        # Show the frame with detections
        cv2.imshow("Detection", frame)

        # Exit if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release resources when done
    cap.release()
    cv2.destroyAllWindows()

# Entry point of the script
if __name__ == "__main__":
    main()
